In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna 
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline, FunctionTransformer 
import statsmodels.api as sm
from sklearn.feature_selection import SequentialFeatureSelector
#from utils.perm_class import ClassificationCV
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression, RidgeClassifier
import pyarrow



c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using 10% of data (stratified & randomized) for EDA, when fitting model will use yield for the whole dataset

In [1]:
from sqlalchemy import create_engine
import pandas as pd

server = '.\SQLEXPRESS' 
database = 'ClubLoanData'

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

engine = create_engine(connection_url)

try:
    query = """SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 1 
    ORDER BY NEWID()) AS Defaults

UNION ALL

SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 0 
    ORDER BY NEWID()) AS GoodLoans
"""
    df = pd.read_sql(query, engine)
    df.to_csv('Data/loan_data_sample.csv')
    print(f"Data Shape: {df.shape}")
except Exception as e:
    print(f"Error: {e}")


c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data Shape: (130639, 91)


In [4]:

df = pd.read_csv('Data\loan_data_sample.csv')
features = ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate',
       'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'total_pymnt', 'total_pymnt_inv',
       'total_rec_prncp', 'total_rec_int', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit', 'Loan_ID',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'sub_grade_num', 'region', 'is_currently_delinq', 'has_il_history']

target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])

#print(df_features.shape,df_predictor.shape)

X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)


OneHotEncoding (essentially)


In [5]:
categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()


X_encoded=pd.get_dummies(X_train[categorical_features],drop_first=True,sparse=False)
X_train = pd.concat([X_train[numerical_features],X_encoded],axis=1)

Correlation / VIF

In [9]:
pd.set_option('display.max_rows', None)
corr = X_train.corrwith(y_train)
corr = corr.abs().sort_values(ascending=False)

corr_df= corr.to_frame(name='Correlation')
print(corr_df)

                                Correlation
total_rec_prncp                    0.441799
total_pymnt                        0.320239
total_pymnt_inv                    0.319754
sub_grade_num                      0.268031
grade                              0.262574
int_rate                           0.259343
term                               0.171540
dti                                0.108633
acc_open_past_24mths               0.104480
num_tl_op_past_12m                 0.094618
all_util                           0.091360
open_rv_24m                        0.086735
verification_status_Verified       0.085443
bc_open_to_buy                     0.084039
avg_cur_bal                        0.078980
open_rv_12m                        0.075677
num_actv_rev_tl                    0.075227
tot_hi_cred_lim                    0.074445
num_rev_tl_bal_gt_0                0.073640
total_bc_limit                     0.073378
mort_acc                           0.072704
inq_last_6mths                  